# Experiment A: First-Order vs Second-Order Trajectory Fitting on GPT-2

**Goal.** Directly compare physics-structured first-order and second-order ODE models
for their ability to predict GPT-2 hidden-state trajectories. This is the leading
item of the experimental agenda (Q1 of paper v4 §19) and the missing Steps 1–5
of the Riemannian programme (§18).

**Key difference from the Markov-order test.** The existing Markov-order regression
(Decision β in the paper) tests whether *generic* lag-2 input adds predictive
power over lag-1. It found lag-1 sufficient (overdamped). This experiment tests
whether *physics-structured* dynamics models with a learned scalar potential V
capture trajectory patterns better when given second-order (position + velocity)
structure vs first-order (position only). The critical evaluation metric is
**multi-step trajectory rollout** R², where velocity coherence matters.

**Four models compared:**
- **M1: First-order physics** — gradient descent on learned V: $h_{t+1} = h_t - \alpha \nabla_h V(h_t)$
- **M2: Second-order physics** — damped Verlet on learned V: uses velocity $v_t = h_t - h_{t-1}$ and damping $\gamma$
- **M3: General lag-1 MLP** — $h_{t+1} = h_t + \text{MLP}(h_t)$, no physics structure
- **M4: General lag-2 MLP** — $h_{t+1} = h_t + \text{MLP}(h_t, v_t)$, no physics structure

**Runtime.** ~20 min on Colab T4 GPU.

In [ ]:
# Cell 1 — Environment setup + GDrive mount
SEED = 42

REPO_URL        = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH     = 'main'
COLAB_REPO_PATH = '/content/semsimula-paper'
GDRIVE_OUT_REL  = 'semsimula_experiment_a'

import os, sys, shutil, subprocess, json, time
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd: str) -> None:
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')


_sh(f'{sys.executable} -m pip install -q transformers torch numpy '
    'scikit-learn matplotlib tqdm scipy')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_OUT = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'GDrive output root = {GDRIVE_OUT}')

    REPO_ROOT = Path(COLAB_REPO_PATH)
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth=1 -b {REPO_BRANCH} {REPO_URL} {COLAB_REPO_PATH}')
    else:
        _sh(f'cd {COLAB_REPO_PATH} && git pull --ff-only')
else:
    REPO_ROOT = Path('__file__').resolve().parents[3]
    if not (REPO_ROOT / 'notebooks').exists():
        REPO_ROOT = Path.cwd()
        while not (REPO_ROOT / 'notebooks').exists() and REPO_ROOT != REPO_ROOT.parent:
            REPO_ROOT = REPO_ROOT.parent
    GDRIVE_OUT = REPO_ROOT / 'notebooks' / 'dynamics_order_test' / 'results' / 'experiment_a'
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)

OUT_DIR = GDRIVE_OUT / f'seed{SEED}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'REPO_ROOT = {REPO_ROOT}')
print(f'SEED      = {SEED}')
print(f'Output    = {OUT_DIR}')

In [ ]:
# Cell 2 — Imports & Configuration
import math, warnings
from dataclasses import dataclass
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=FutureWarning)

CFG = dict(
    model_name      = 'gpt2',
    layer            = -1,         # last transformer layer
    d_pca            = 50,         # PCA projection dimension
    n_train_sent     = 40,         # leave 10 for test
    d_hidden_V       = 128,        # hidden dim of potential MLP
    n_layers_V       = 2,          # depth of potential MLP
    d_hidden_gen     = 128,
    n_layers_gen     = 2,
    lr               = 1e-3,
    n_epochs         = 400,
    batch_size       = 256,
    weight_decay     = 1e-4,
    rollout_steps    = [1, 3, 5, 8, 10],
    seed             = SEED,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# Save config
with open(OUT_DIR / 'config.json', 'w') as f:
    json.dump(CFG, f, indent=2)
print(f'Config saved to {OUT_DIR / "config.json"}')
print(f'Config: {CFG}')

In [ ]:
# Cell 3 — Corpus (same 50-sentence, 5-domain corpus as the paper)
CORPUS = {
  "mathematics": [
    "The fundamental theorem of calculus establishes that differentiation and integration are inverse operations of each other.",
    "A metric space is a set together with a notion of distance between its elements, usually called points, that satisfies a set of axioms.",
    "Euler's identity connects the five most important numbers in mathematics through the equation e to the power of i pi plus one equals zero.",
    "The eigenvalues of a symmetric matrix are always real, and the eigenvectors corresponding to distinct eigenvalues are orthogonal.",
    "Gödel's incompleteness theorems demonstrate that in any consistent formal system capable of expressing basic arithmetic there exist statements that can neither be proved nor disproved.",
    "The Riemann hypothesis conjectures that all non-trivial zeros of the Riemann zeta function have real part equal to one half.",
    "A group homomorphism preserves the algebraic structure by mapping the identity element to the identity element and products to products.",
    "The central limit theorem states that the sum of a large number of independent random variables tends toward a normal distribution regardless of the underlying distribution.",
    "Hilbert spaces generalize the notion of Euclidean space to infinite dimensions while retaining the structure of an inner product.",
    "The Lagrangian of a mechanical system equals the kinetic energy minus the potential energy and encodes the complete dynamics through the Euler-Lagrange equations."
  ],
  "narrative": [
    "The old lighthouse keeper climbed the spiral staircase one last time, his weathered hands gripping the iron railing as the storm gathered outside.",
    "She found the letter tucked between the pages of a book she hadn't opened in years, the ink faded but the words still sharp enough to wound.",
    "The train pulled into the empty station at midnight, its headlamp cutting through the fog like a single unblinking eye.",
    "He sat on the porch watching the fireflies trace their erratic paths through the warm summer air while the radio played something slow and sad.",
    "The market was closing for the day and the vendors were packing up their unsold fruit, bruised peaches and overripe plums going back into crates.",
    "She ran through the forest with branches whipping at her face, the sound of the river growing louder with every desperate step.",
    "The children built a fort out of couch cushions and draped a bedsheet over the top, declaring it a castle that no adults could enter.",
    "He returned to the village after twenty years and found that the oak tree in the square had been cut down and replaced by a parking lot.",
    "The ship appeared on the horizon at dawn, its sails torn and its hull battered, carrying survivors of a voyage no one had expected to end.",
    "She opened the old music box and it played the same melody her grandmother used to hum while braiding her hair on Sunday mornings."
  ],
  "scientific": [
    "Photosynthesis converts carbon dioxide and water into glucose and oxygen using light energy absorbed by chlorophyll molecules in the thylakoid membranes.",
    "The theory of plate tectonics explains that Earth's lithosphere is divided into several large plates that float on the semi-fluid asthenosphere beneath.",
    "Antibiotics work by either killing bacteria directly or inhibiting their ability to grow and reproduce, but they have no effect on viral infections.",
    "Black holes are regions of spacetime where gravity is so strong that nothing, not even light, can escape once it crosses the event horizon.",
    "The human genome contains approximately three billion base pairs of DNA organized into twenty-three pairs of chromosomes in each cell nucleus.",
    "Quantum entanglement describes a phenomenon where two particles become correlated in such a way that the quantum state of each particle cannot be described independently.",
    "Mitochondria are often called the powerhouses of the cell because they generate most of the cell's supply of adenosine triphosphate used as chemical energy.",
    "The Doppler effect explains why the pitch of an ambulance siren appears to change as the vehicle approaches and then recedes from an observer.",
    "CRISPR-Cas9 is a gene-editing technology that allows scientists to add, remove, or alter genetic material at particular locations in the genome with unprecedented precision.",
    "Dark matter makes up approximately twenty-seven percent of the universe's total mass-energy content but does not interact with electromagnetic radiation and has never been directly observed."
  ],
  "code_description": [
    "A hash table stores key-value pairs and uses a hash function to compute an index into an array of buckets from which the desired value can be found.",
    "Recursion is a technique where a function calls itself with a modified argument until it reaches a base case that returns without further recursive calls.",
    "The model-view-controller pattern separates an application into three interconnected components to separate internal representations from the ways information is presented.",
    "Gradient descent is an optimization algorithm that iteratively adjusts parameters in the direction of steepest descent of the loss function to find a local minimum.",
    "A binary search tree maintains sorted data and allows lookup, insertion, and deletion operations in time proportional to the logarithm of the number of elements.",
    "Docker containers package an application with all its dependencies into a standardized unit that can run consistently across different computing environments.",
    "Backpropagation computes the gradient of the loss function with respect to each weight by applying the chain rule layer by layer from the output back to the input.",
    "A relational database organizes data into tables of rows and columns and uses structured query language to manage and retrieve the stored information.",
    "Version control systems like Git track changes to source code over time, allowing multiple developers to collaborate on a project without overwriting each other's work.",
    "An API defines a set of rules and protocols that allows different software applications to communicate with each other by sending requests and receiving responses."
  ],
  "conversational": [
    "I think we should grab coffee sometime this week because there are a few things I've been meaning to discuss with you about the project timeline.",
    "Have you ever noticed how the same song can sound completely different depending on whether you're happy or sad when you hear it?",
    "My neighbor's dog escaped again last night and we spent two hours searching the neighborhood before finding him asleep under a parked car.",
    "The restaurant on the corner finally reopened after the renovation and honestly the food is even better now than it was before they closed.",
    "I tried to learn how to play the piano when I was a kid but gave up after about six months because I couldn't stand practicing scales.",
    "She asked me what I wanted for my birthday and I couldn't think of a single thing which made me realize I already have everything I need.",
    "The weather forecast said it would rain all weekend but the sun came out Saturday morning and stayed until Monday.",
    "I'm not sure if I should take the job offer because it pays more but the commute would add two hours to my day.",
    "We watched three movies in a row last night and by the end of the third one everyone had fallen asleep on the couch.",
    "He told me the secret to his grandmother's pasta sauce is a pinch of cinnamon which sounds strange but actually makes a big difference."
  ]
}

sentences, domains = [], []
for domain, sents in CORPUS.items():
    for s in sents:
        sentences.append(s)
        domains.append(domain)
print(f'Corpus: {len(sentences)} sentences, {len(set(domains))} domains')

In [ ]:
# Cell 4 — Extract hidden states from GPT-2
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
model = AutoModelForCausalLM.from_pretrained(
    CFG['model_name'], output_hidden_states=True
).eval().to(DEVICE)

@torch.no_grad()
def extract_hidden_states(sentence: str, layer: int = -1) -> np.ndarray:
    """Return (T, d_model) hidden states at the given layer."""
    toks = tokenizer(sentence, return_tensors='pt').to(DEVICE)
    out = model(**toks)
    hs = out.hidden_states[layer]  # (1, T, d_model)
    return hs.squeeze(0).cpu().numpy()

all_hidden = []  # list of (T_i, d_model) arrays
for i, s in enumerate(tqdm(sentences, desc='Extracting hidden states')):
    hs = extract_hidden_states(s, layer=CFG['layer'])
    all_hidden.append(hs)

total_tokens = sum(h.shape[0] for h in all_hidden)
d_model = all_hidden[0].shape[1]
print(f'Extracted {total_tokens} tokens across {len(all_hidden)} sentences, d_model={d_model}')

del model  # free GPU memory
torch.cuda.empty_cache() if DEVICE == 'cuda' else None

In [ ]:
# Cell 5 — PCA reduction
d_pca = CFG['d_pca']

all_vectors = np.concatenate(all_hidden, axis=0)  # (N_total, d_model)
pca = PCA(n_components=d_pca, random_state=CFG['seed'])
pca.fit(all_vectors)
explained = pca.explained_variance_ratio_.sum()
print(f'PCA-{d_pca} explains {explained:.1%} of variance')

hidden_pca = [pca.transform(h) for h in all_hidden]  # list of (T_i, d_pca)
del all_vectors, all_hidden

In [ ]:
# Cell 6 — Prepare triplets and trajectory data
# For training: triplets (h_{t-1}, h_t, h_{t+1}) — used by all models
# For multi-step rollout: full sentence trajectories

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])

n_train = CFG['n_train_sent']
perm = np.random.permutation(len(sentences))
train_idx = perm[:n_train]
test_idx = perm[n_train:]

def make_triplets(indices):
    """Create arrays of (h_{t-1}, h_t, h_{t+1}) from sentence indices."""
    h_prev, h_curr, h_next, sent_ids = [], [], [], []
    for i in indices:
        h = hidden_pca[i]  # (T, d_pca)
        T = h.shape[0]
        if T < 3:
            continue
        for t in range(1, T - 1):
            h_prev.append(h[t-1])
            h_curr.append(h[t])
            h_next.append(h[t+1])
            sent_ids.append(i)
    return (
        torch.tensor(np.array(h_prev), dtype=torch.float32),
        torch.tensor(np.array(h_curr), dtype=torch.float32),
        torch.tensor(np.array(h_next), dtype=torch.float32),
        np.array(sent_ids),
    )

hp_train, hc_train, hn_train, sid_train = make_triplets(train_idx)
hp_test,  hc_test,  hn_test,  sid_test  = make_triplets(test_idx)

print(f'Train: {hp_train.shape[0]} triplets from {n_train} sentences')
print(f'Test:  {hp_test.shape[0]} triplets from {len(test_idx)} sentences')

In [ ]:
# Cell 7 — Define the four dynamics models

class ScalarPotentialMLP(nn.Module):
    """MLP V : R^d -> R as a learned scalar potential."""
    def __init__(self, d_in, d_hidden, n_layers):
        super().__init__()
        layers = []
        d = d_in
        for _ in range(n_layers):
            layers += [nn.Linear(d, d_hidden), nn.GELU()]
            d = d_hidden
        layers.append(nn.Linear(d, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, h):
        return self.net(h).squeeze(-1)


class FirstOrderPhysics(nn.Module):
    """M1: h_{t+1} = h_t - alpha * grad_h V(h_t).

    Pure gradient descent on a learned scalar potential.
    Predicts h_{t+1} from h_t only (no velocity).
    """
    def __init__(self, d, d_hidden, n_layers):
        super().__init__()
        self.V = ScalarPotentialMLP(d, d_hidden, n_layers)
        self.log_alpha = nn.Parameter(torch.tensor(0.0))

    def forward(self, h_curr, h_prev=None):
        h = h_curr.detach().requires_grad_(True)
        V = self.V(h)
        grad_V = torch.autograd.grad(V.sum(), h, create_graph=True)[0]
        alpha = self.log_alpha.exp()
        return h_curr - alpha * grad_V

    def rollout(self, h0, K):
        """Unroll K steps from initial position h0."""
        traj = [h0]
        h = h0
        for _ in range(K):
            h = self.forward(h)
            traj.append(h)
        return torch.stack(traj, dim=1)  # (B, K+1, d)


class SecondOrderPhysics(nn.Module):
    """M2: Damped Velocity-Verlet on a learned scalar potential.

    h_{t+1} = h_t + v_t / (1 + gamma) - alpha / (1 + gamma) * grad_h V(h_t)
    Uses velocity v_t = h_t - h_{t-1} and learnable damping gamma.
    """
    def __init__(self, d, d_hidden, n_layers):
        super().__init__()
        self.V = ScalarPotentialMLP(d, d_hidden, n_layers)
        self.log_alpha = nn.Parameter(torch.tensor(0.0))
        self.log_gamma = nn.Parameter(torch.tensor(0.0))  # init gamma=1

    def step(self, h_curr, v_curr):
        h = h_curr.detach().requires_grad_(True)
        V = self.V(h)
        grad_V = torch.autograd.grad(V.sum(), h, create_graph=True)[0]
        alpha = self.log_alpha.exp()
        gamma = self.log_gamma.exp()
        damp = 1.0 / (1.0 + gamma)
        h_next = h_curr + damp * v_curr - damp * alpha * grad_V
        v_next = h_next - h_curr
        return h_next, v_next

    def forward(self, h_curr, h_prev):
        v = h_curr - h_prev
        h_next, _ = self.step(h_curr, v)
        return h_next

    def rollout(self, h0, h1, K):
        """Unroll K steps given h0 and h1 (two initial positions)."""
        traj = [h0, h1]
        h, v = h1, h1 - h0
        for _ in range(K - 1):
            h, v = self.step(h, v)
            traj.append(h)
        return torch.stack(traj, dim=1)  # (B, K+1, d)


class GeneralMLPDynamics(nn.Module):
    """General MLP displacement predictor (no physics structure).

    h_{t+1} = h_t + MLP(input)
    If use_velocity=False (M3): input = h_t       (lag-1)
    If use_velocity=True  (M4): input = [h_t; v_t] (lag-2)
    """
    def __init__(self, d, d_hidden, n_layers, use_velocity=False):
        super().__init__()
        self.use_velocity = use_velocity
        d_in = 2 * d if use_velocity else d
        layers = []
        din = d_in
        for _ in range(n_layers):
            layers += [nn.Linear(din, d_hidden), nn.GELU()]
            din = d_hidden
        layers.append(nn.Linear(din, d))
        self.net = nn.Sequential(*layers)

    def forward(self, h_curr, h_prev=None):
        if self.use_velocity:
            v = h_curr - h_prev
            x = torch.cat([h_curr, v], dim=-1)
        else:
            x = h_curr
        return h_curr + self.net(x)

    def rollout(self, h0, K, h_minus1=None):
        """Unroll K steps."""
        traj = [h0]
        h = h0
        h_prev = h_minus1 if h_minus1 is not None else h0
        for _ in range(K):
            h_new = self.forward(h, h_prev)
            h_prev = h
            h = h_new
            traj.append(h)
        return torch.stack(traj, dim=1)


d = CFG['d_pca']
models = {
    'M1_first_order_physics':  FirstOrderPhysics(d, CFG['d_hidden_V'], CFG['n_layers_V']),
    'M2_second_order_physics': SecondOrderPhysics(d, CFG['d_hidden_V'], CFG['n_layers_V']),
    'M3_general_lag1_mlp':     GeneralMLPDynamics(d, CFG['d_hidden_gen'], CFG['n_layers_gen'], use_velocity=False),
    'M4_general_lag2_mlp':     GeneralMLPDynamics(d, CFG['d_hidden_gen'], CFG['n_layers_gen'], use_velocity=True),
}

for name, m in models.items():
    n_params = sum(p.numel() for p in m.parameters())
    print(f'{name}: {n_params:,} parameters')

In [ ]:
# Cell 8 — Training loop (saves logs + checkpoints to GDrive)

def train_model(name, model, hp_train, hc_train, hn_train, cfg, out_dir):
    """Train a dynamics model to predict h_{t+1} from (h_{t-1}, h_t)."""
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'],
                            weight_decay=cfg['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg['n_epochs'])

    hp = hp_train.to(DEVICE)
    hc = hc_train.to(DEVICE)
    hn = hn_train.to(DEVICE)
    N = hc.shape[0]
    bs = cfg['batch_size']

    model_dir = out_dir / name
    model_dir.mkdir(parents=True, exist_ok=True)
    log_path = model_dir / 'training_log.jsonl'

    losses = []
    t0 = time.time()
    best_loss = float('inf')

    with open(log_path, 'w') as log_f:
        for epoch in range(cfg['n_epochs']):
            model.train()
            perm = torch.randperm(N, device=DEVICE)
            epoch_loss = 0.0
            n_batches = 0

            for start in range(0, N, bs):
                idx = perm[start:start+bs]
                hp_b = hp[idx]
                hc_b = hc[idx]
                hn_b = hn[idx]

                pred = model(hc_b, hp_b)
                loss = F.mse_loss(pred, hn_b)

                opt.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

                epoch_loss += loss.item()
                n_batches += 1

            scheduler.step()
            avg = epoch_loss / n_batches
            losses.append(avg)

            record = {'epoch': epoch + 1, 'train_mse': avg,
                      'elapsed_s': time.time() - t0}
            log_f.write(json.dumps(record) + '\n')

            if avg < best_loss:
                best_loss = avg
                torch.save({
                    'epoch': epoch + 1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': opt.state_dict(),
                    'train_mse': avg,
                }, model_dir / 'best_model.pt')

            if (epoch + 1) % 100 == 0:
                log_f.flush()
                print(f'  [{name}] epoch {epoch+1}/{cfg["n_epochs"]}: '
                      f'train MSE = {avg:.6f}')

    elapsed = time.time() - t0
    print(f'  [{name}] Done in {elapsed:.1f}s, final train MSE = {losses[-1]:.6f}')
    print(f'  Logs: {log_path}')
    print(f'  Best checkpoint: {model_dir / "best_model.pt"}')
    return model, losses

trained_models = {}
train_losses = {}

for name, m in models.items():
    print(f'\nTraining {name}...')
    trained_m, losses = train_model(name, m, hp_train, hc_train, hn_train, CFG, OUT_DIR)
    trained_models[name] = trained_m
    train_losses[name] = losses

In [ ]:
# Cell 9 — Single-step evaluation (R² on held-out test triplets)

def compute_r2(y_true, y_pred):
    """Multivariate R²: 1 - SS_res / SS_tot."""
    ss_res = ((y_true - y_pred) ** 2).sum()
    ss_tot = ((y_true - y_true.mean(dim=0, keepdim=True)) ** 2).sum()
    return 1.0 - (ss_res / ss_tot).item()

def eval_single_step(model, hp, hc, hn):
    model.eval()
    hp_d = hp.to(DEVICE)
    hc_d = hc.to(DEVICE)
    hn_d = hn.to(DEVICE)
    with torch.enable_grad():
        pred = model(hc_d, hp_d)
    mse = F.mse_loss(pred.detach(), hn_d).item()
    r2 = compute_r2(hn_d, pred.detach())
    return mse, r2

print('=== Single-step prediction (test set) ===')
print(f'{"Model":<30s} {"MSE":>10s} {"R²":>10s}')
print('-' * 52)
single_step_results = {}
for name, m in trained_models.items():
    mse, r2 = eval_single_step(m, hp_test, hc_test, hn_test)
    single_step_results[name] = {'mse': mse, 'r2': r2}
    print(f'{name:<30s} {mse:>10.6f} {r2:>10.6f}')

with open(OUT_DIR / 'single_step_results.json', 'w') as f:
    json.dump(single_step_results, f, indent=2)
print(f'\nSaved to {OUT_DIR / "single_step_results.json"}')

In [ ]:
# Cell 10 — Multi-step rollout evaluation
# For each test sentence, unroll K steps from initial conditions
# and compute R² at each step horizon.

def eval_rollout_per_sentence(model, name, sentence_idx, hidden_pca_list, K_max):
    """Evaluate multi-step rollout R² for one sentence."""
    model.eval()
    h_full = torch.tensor(hidden_pca_list[sentence_idx], dtype=torch.float32,
                          device=DEVICE)  # (T, d)
    T = h_full.shape[0]
    if T < K_max + 2:
        return None

    results = {}  # k -> {mse, r2}
    is_second_order = 'second_order' in name or 'lag2' in name

    for K in CFG['rollout_steps']:
        if K + 2 > T:
            continue

        preds = []
        trues = []

        n_starts = T - K - 1 if is_second_order else T - K
        start_min = 1 if is_second_order else 0

        for t0 in range(start_min, min(start_min + n_starts, T - K)):
            h0 = h_full[t0].unsqueeze(0)
            true_traj = h_full[t0:t0+K+1]  # (K+1, d)

            with torch.enable_grad():
                if 'first_order_physics' in name:
                    pred_traj = model.rollout(h0, K).squeeze(0)
                elif 'second_order_physics' in name:
                    h_m1 = h_full[t0-1].unsqueeze(0)
                    pred_traj = model.rollout(h_m1, h0, K).squeeze(0)
                elif 'lag2' in name:
                    h_m1 = h_full[t0-1].unsqueeze(0)
                    pred_traj = model.rollout(h0, K, h_minus1=h_m1).squeeze(0)
                else:  # lag1
                    pred_traj = model.rollout(h0, K).squeeze(0)

            pred_final = pred_traj[-1].detach()
            true_final = true_traj[-1]
            preds.append(pred_final)
            trues.append(true_final)

        if len(preds) == 0:
            continue
        preds_t = torch.stack(preds)
        trues_t = torch.stack(trues)
        mse = F.mse_loss(preds_t, trues_t).item()
        r2 = compute_r2(trues_t, preds_t)
        results[K] = {'mse': mse, 'r2': r2}

    return results

print('\n=== Multi-step rollout evaluation (test sentences) ===')

K_max = max(CFG['rollout_steps'])
rollout_results = {name: {K: [] for K in CFG['rollout_steps']}
                   for name in trained_models}

for si in test_idx:
    for name, m in trained_models.items():
        res = eval_rollout_per_sentence(m, name, si, hidden_pca, K_max)
        if res is None:
            continue
        for K, vals in res.items():
            rollout_results[name][K].append(vals)

# Aggregate
print(f'\n{"Model":<30s}', end='')
for K in CFG['rollout_steps']:
    print(f' {"K="+str(K):>8s}', end='')
print()
print('-' * (30 + 9 * len(CFG['rollout_steps'])))

rollout_r2_means = {}
for name in trained_models:
    rollout_r2_means[name] = {}
    print(f'{name:<30s}', end='')
    for K in CFG['rollout_steps']:
        vals = rollout_results[name][K]
        if vals:
            mean_r2 = np.mean([v['r2'] for v in vals])
            rollout_r2_means[name][K] = mean_r2
            print(f' {mean_r2:>8.4f}', end='')
        else:
            print(f' {"N/A":>8s}', end='')
    print()

# Save rollout results
rollout_save = {}
for name in trained_models:
    rollout_save[name] = {
        str(K): {'mean_r2': rollout_r2_means[name].get(K),
                 'per_sentence': [v for v in rollout_results[name][K]]}
        for K in CFG['rollout_steps']
    }
with open(OUT_DIR / 'rollout_results.json', 'w') as f:
    json.dump(rollout_save, f, indent=2)
print(f'\nSaved to {OUT_DIR / "rollout_results.json"}')

In [ ]:
# Cell 11 — Training loss curves

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
for name, losses in train_losses.items():
    ax.plot(losses, label=name, alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Train MSE')
ax.set_yscale('log')
ax.set_title('Training Loss Curves')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(OUT_DIR / 'training_loss_curves.png', dpi=150, bbox_inches='tight')
print(f'Saved to {OUT_DIR / "training_loss_curves.png"}')
plt.show()

In [ ]:
# Cell 12 — Multi-step R² bar chart

fig, axes = plt.subplots(1, len(CFG['rollout_steps']), figsize=(4*len(CFG['rollout_steps']), 5),
                         sharey=True)
if len(CFG['rollout_steps']) == 1:
    axes = [axes]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
model_names = list(trained_models.keys())
short_names = ['M1: 1st-order\nphysics', 'M2: 2nd-order\nphysics',
               'M3: General\nlag-1 MLP', 'M4: General\nlag-2 MLP']

for ki, K in enumerate(CFG['rollout_steps']):
    ax = axes[ki]
    vals = [rollout_r2_means.get(n, {}).get(K, float('nan')) for n in model_names]
    bars = ax.bar(range(4), vals, color=colors, alpha=0.8)
    ax.set_xticks(range(4))
    ax.set_xticklabels(short_names, fontsize=7)
    ax.set_title(f'K={K} step rollout', fontsize=11)
    if ki == 0:
        ax.set_ylabel('R² (test)', fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    for i, v in enumerate(vals):
        if not np.isnan(v):
            ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=8)

fig.suptitle('Multi-step Rollout R² — First-Order vs Second-Order Dynamics on GPT-2',
             fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(OUT_DIR / 'rollout_r2_bar_chart.png', dpi=150, bbox_inches='tight')
print(f'Saved to {OUT_DIR / "rollout_r2_bar_chart.png"}')
plt.show()

In [ ]:
# Cell 13 — R² decay curves (R² vs rollout horizon K)

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
for i, name in enumerate(model_names):
    Ks = sorted(rollout_r2_means.get(name, {}).keys())
    r2s = [rollout_r2_means[name][K] for K in Ks]
    ax.plot(Ks, r2s, 'o-', color=colors[i], label=short_names[i].replace('\n', ' '),
            linewidth=2, markersize=6)

ax.set_xlabel('Rollout horizon K (steps)', fontsize=12)
ax.set_ylabel('Mean R² on test sentences', fontsize=12)
ax.set_title('R² Decay with Rollout Horizon — Physics vs General Models', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=-0.1)
plt.tight_layout()
fig.savefig(OUT_DIR / 'r2_decay_curves.png', dpi=150, bbox_inches='tight')
print(f'Saved to {OUT_DIR / "r2_decay_curves.png"}')
plt.show()

In [ ]:
# Cell 14 — Example trajectory overlay (one test sentence)

best_test_sent = None
best_len = 0
for si in test_idx:
    T = hidden_pca[si].shape[0]
    if T > best_len:
        best_len = T
        best_test_sent = si

h_true = torch.tensor(hidden_pca[best_test_sent], dtype=torch.float32,
                       device=DEVICE)
T = h_true.shape[0]
print(f'Visualizing sentence {best_test_sent} (T={T} tokens): '
      f'"{sentences[best_test_sent][:80]}..."')

# PCA-2 of this sentence's trajectory for visualization
from sklearn.decomposition import PCA as PCA2
pca2 = PCA2(n_components=2)
h_true_np = h_true.cpu().numpy()
h_2d_true = pca2.fit_transform(h_true_np)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
K_vis = min(T - 2, 10)

for idx, (name, m) in enumerate(trained_models.items()):
    ax = axes[idx // 2, idx % 2]
    m.eval()

    with torch.enable_grad():
        h0 = h_true[1].unsqueeze(0)
        if 'first_order_physics' in name:
            pred_traj = m.rollout(h0, K_vis).squeeze(0)
        elif 'second_order_physics' in name:
            hm1 = h_true[0].unsqueeze(0)
            pred_traj = m.rollout(hm1, h0, K_vis).squeeze(0)
        elif 'lag2' in name:
            hm1 = h_true[0].unsqueeze(0)
            pred_traj = m.rollout(h0, K_vis, h_minus1=hm1).squeeze(0)
        else:
            pred_traj = m.rollout(h0, K_vis).squeeze(0)

    pred_np = pred_traj.detach().cpu().numpy()
    pred_2d = pca2.transform(pred_np)

    ax.plot(h_2d_true[:K_vis+2, 0], h_2d_true[:K_vis+2, 1],
            'k-o', markersize=4, alpha=0.5, label='True (GPT-2)', linewidth=1.5)
    ax.plot(pred_2d[:, 0], pred_2d[:, 1],
            's-', markersize=4, alpha=0.7, label=f'Predicted ({name})', linewidth=1.5)
    ax.plot(h_2d_true[1, 0], h_2d_true[1, 1], 'g*', markersize=12, label='Start')
    ax.set_title(short_names[idx].replace('\n', ' '), fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

fig.suptitle(f'Trajectory Rollout (K={K_vis}) in PCA-2 Space', fontsize=14, y=1.01)
plt.tight_layout()
fig.savefig(OUT_DIR / 'trajectory_overlay_pca2.png', dpi=150, bbox_inches='tight')
print(f'Saved to {OUT_DIR / "trajectory_overlay_pca2.png"}')
plt.show()

In [ ]:
# Cell 15 — Learned dynamics parameters

print('=== Learned dynamics parameters ===')
learned_params = {}
for name, m in trained_models.items():
    params = {}
    print(f'\n{name}:')
    if hasattr(m, 'log_alpha'):
        alpha = m.log_alpha.exp().item()
        params['alpha'] = alpha
        print(f'  alpha (step size) = {alpha:.6f}')
    if hasattr(m, 'log_gamma'):
        gamma = m.log_gamma.exp().item()
        params['gamma'] = gamma
        params['velocity_retention'] = 1.0 / (1.0 + gamma)
        print(f'  gamma (damping)   = {gamma:.6f}')
        print(f'  1/(1+gamma)       = {1/(1+gamma):.6f}  (velocity retention)')
    learned_params[name] = params

with open(OUT_DIR / 'learned_params.json', 'w') as f:
    json.dump(learned_params, f, indent=2)
print(f'\nSaved to {OUT_DIR / "learned_params.json"}')

In [ ]:
# Cell 16 — Statistical significance: paired Wilcoxon on per-sentence R²
from scipy.stats import wilcoxon

def paired_comparison(name_a, name_b, K):
    """Compare two models' per-sentence R² at rollout K."""
    vals_a = rollout_results[name_a][K]
    vals_b = rollout_results[name_b][K]
    n = min(len(vals_a), len(vals_b))
    if n < 5:
        return None
    r2_a = np.array([v['r2'] for v in vals_a[:n]])
    r2_b = np.array([v['r2'] for v in vals_b[:n]])
    diff = r2_a - r2_b
    mean_diff = diff.mean()
    try:
        stat, p = wilcoxon(diff, alternative='two-sided')
    except ValueError:
        stat, p = float('nan'), float('nan')
    sign_cons = (diff > 0).sum()
    return {'mean_diff': mean_diff, 'p': p, 'sign_positive': f'{sign_cons}/{n}',
            'n': n}

print('=== Paired comparisons (M2 vs M1, M4 vs M3) ===')
comparisons = [
    ('M2_second_order_physics', 'M1_first_order_physics', 'M2 vs M1 (physics)'),
    ('M4_general_lag2_mlp', 'M3_general_lag1_mlp', 'M4 vs M3 (general)'),
    ('M2_second_order_physics', 'M4_general_lag2_mlp', 'M2 vs M4 (physics vs general)'),
]

stat_results = {}
for name_a, name_b, label in comparisons:
    print(f'\n--- {label} ---')
    stat_results[label] = {}
    for K in CFG['rollout_steps']:
        res = paired_comparison(name_a, name_b, K)
        if res:
            stat_results[label][f'K={K}'] = res
            print(f'  K={K:>2d}: ΔR² = {res["mean_diff"]:+.4f}, '
                  f'p = {res["p"]:.4f}, sign = {res["sign_positive"]}')
        else:
            print(f'  K={K:>2d}: insufficient data')

with open(OUT_DIR / 'statistical_tests.json', 'w') as f:
    json.dump(stat_results, f, indent=2)
print(f'\nSaved to {OUT_DIR / "statistical_tests.json"}')

In [ ]:
# Cell 17 — Summary & Interpretation (saved to GDrive)

print('=' * 70)
print('EXPERIMENT A — SUMMARY')
print('=' * 70)
print()
print('1. SINGLE-STEP PREDICTION (K=1):')
for name in model_names:
    r = single_step_results[name]
    print(f'   {name:<30s}  R² = {r["r2"]:.4f}  MSE = {r["mse"]:.6f}')

print()
print('2. MULTI-STEP ROLLOUT R² (key horizon):')
K_key = 5 if 5 in CFG['rollout_steps'] else CFG['rollout_steps'][-1]
for name in model_names:
    r2 = rollout_r2_means.get(name, {}).get(K_key, float('nan'))
    print(f'   {name:<30s}  R²(K={K_key}) = {r2:.4f}')

print()
print('3. INTERPRETATION:')
print('   - If M2 >> M1 at large K: second-order physics structure captures')
print('     trajectory-scale coherence that first-order gradient flow misses.')
print('   - If M1 ≈ M2 at K=1 but M2 >> M1 at K=5+: the single-step dynamics')
print('     looks first-order (consistent with Decision β) but the multi-step')
print('     trajectory requires second-order velocity coherence.')
print('   - If M4 >> M3: velocity information is generically useful (lag-2 > lag-1).')
print('   - If M2 >> M4: physics-structured second-order dynamics captures')
print('     patterns that a generic lag-2 MLP cannot, validating the Lagrangian')
print('     framework\'s inductive bias.')
print()
print('4. LEARNED DAMPING (M2):')
gamma_val = None
if 'log_gamma' in [n for n, _ in trained_models['M2_second_order_physics'].named_parameters()]:
    gamma_val = trained_models['M2_second_order_physics'].log_gamma.exp().item()
    print(f'   gamma = {gamma_val:.4f}')
    if gamma_val > 5:
        print('   -> Large gamma: dynamics is effectively overdamped (consistent with')
        print('      Decision β and the paper\'s overdamped synthesis).')
    elif gamma_val < 0.5:
        print('   -> Small gamma: dynamics retains substantial inertia. This would')
        print('      provide direct evidence for second-order dynamics in GPT-2.')
    else:
        print(f'   -> Moderate gamma: intermediate damping regime.')

# Save full summary
summary = {
    'single_step': single_step_results,
    'rollout_r2_means': {name: {str(k): v for k, v in d.items()}
                         for name, d in rollout_r2_means.items()},
    'learned_gamma': gamma_val,
    'pca_variance_explained': float(explained),
    'n_train_triplets': int(hp_train.shape[0]),
    'n_test_triplets': int(hp_test.shape[0]),
    'n_train_sentences': int(CFG['n_train_sent']),
    'n_test_sentences': int(len(test_idx)),
    'config': CFG,
}
with open(OUT_DIR / 'experiment_a_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\n{"=" * 70}')
print(f'All results saved to: {OUT_DIR}')
print(f'Files:')
for p in sorted(OUT_DIR.rglob('*')):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f'  {p.relative_to(OUT_DIR)}  ({size_kb:.1f} KB)')
print(f'{"=" * 70}')